# 🧠 Benchmark 4 — MIND (embeddings de phrases en français)

**MIND v2** : `Geotrend/distilbert-base-en-fr-cased` (68,8 M paramètres) + mean pooling + Dense 768→256 (tanh) + normalisation,
entraîné avec MultipleNegativesRankingLoss sur 81 829 paires (PAWS-X fr + XNLI fr).

On le compare à des modèles de référence de taille proche :

| Modèle | Paramètres | Dim | Remarque |
|---|---|---|---|
| **MIND v2** | 69 M | 256 | le tien |
| Geotrend/distilbert-base-en-fr-cased | 69 M | 768 | le modèle de départ, **sans** fine-tuning → mesure ce que ton entraînement apporte |
| sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 | 118 M | 384 | classique multilingue |
| sentence-transformers/distiluse-base-multilingual-cased-v2 | 135 M | 512 | DistilBERT multilingue, architecture la plus proche |
| intfloat/multilingual-e5-small | 118 M | 384 | fort pour sa taille |
| dangvantuan/sentence-camembert-base | 110 M | 768 | spécialiste du français |

**Partie A** — tests rapides maison : STSb-fr, STS croisé fr↔en, vitesse.
**Partie B** — **MTEB français** (`MTEB(fra, v1)`, 25 tâches : classification, clustering, paires, reranking, recherche, STS, résumé),
le benchmark de référence pour les embeddings français.

⚠️ `PawsXPairClassification` (et XNLI) sont **dans le domaine d'entraînement** de MIND : c'est signalé dans les résultats.

⚙️ Réglages Kaggle : **Internet ON**, **GPU T4**. Durée : partie A ~10 min, partie B ~2–4 h (mode `LEGER` ~30 min).

In [ ]:
!pip install -q -U sentence-transformers mteb

In [ ]:
import os, glob, time, gc, urllib.request
import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
from sentence_transformers import SentenceTransformer
import mteb
from scipy.stats import spearmanr, pearsonr

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SORTIE = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
print("device :", DEVICE, "| mteb", mteb.__version__)

# ---- réglages ----
LEGER = False          # True : quelques tâches MTEB seulement (~30 min)
BASELINES = [
    "Geotrend/distilbert-base-en-fr-cased",
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    "sentence-transformers/distiluse-base-multilingual-cased-v2",
    "intfloat/multilingual-e5-small",
    "dangvantuan/sentence-camembert-base",
    # "intfloat/multilingual-e5-base", "BAAI/bge-m3",   # plus gros, si tu veux viser haut
]
PREFIXES = {"intfloat/multilingual-e5-small": "query: ", "intfloat/multilingual-e5-base": "query: "}

## Téléchargement de MIND
Cherché d'abord dans `/kaggle/input` (si tu as ajouté le dossier `mind-v2-final` comme dataset), sinon téléchargé depuis GitHub.

In [ ]:
FICHIERS_MIND = ["README.md", "config.json", "config_sentence_transformers.json", "modules.json",
                 "sentence_bert_config.json", "tokenizer.json", "tokenizer_config.json", "model.safetensors",
                 "1_Pooling/config.json", "2_Dense/config.json", "2_Dense/model.safetensors", "3_Normalize/config.json"]
LFS = "https://media.githubusercontent.com/media/LiamLitle/ia-de-liam/main/MIND/mind-v2-final/"

trouve = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/modules.json", recursive=True)
          if os.path.exists(os.path.join(os.path.dirname(p), "2_Dense"))]
if trouve:
    MIND = trouve[0]
else:
    MIND = f"{SORTIE}/mind-v2-final"
    for f in FICHIERS_MIND:
        dest = os.path.join(MIND, f)
        os.makedirs(os.path.dirname(dest), exist_ok=True)
        if not os.path.exists(dest):
            urllib.request.urlretrieve(LFS + f, dest)
print("MIND :", MIND)
mind = SentenceTransformer(MIND, device=DEVICE)
mind.model_card_data.model_name = "LiamLitle/mind-v2"  # nom utilisé par MTEB pour son cache de résultats
print(mind)
e = mind.encode(["Le chat dort sur le canapé.", "Un chat fait la sieste sur le sofa.", "La bourse a chuté hier."])
print(np.round(e @ e.T, 3))

## Partie A — tests rapides
### A1. STSb-fr (test) et STS croisé français ↔ anglais
`stsb_multi_mt` contient les **mêmes paires traduites** : on peut comparer une phrase française à une phrase anglaise.
Le test STSb-fr n'a pas servi à l'entraînement de MIND (seul le dev a servi à choisir le checkpoint) → le README annonce **68,2** (Spearman ×100).

In [ ]:
from huggingface_hub import hf_hub_download
def stsb(langue):
    return pd.read_parquet(hf_hub_download("PhilipMay/stsb_multi_mt", f"{langue}/test-00000-of-00001.parquet", repo_type="dataset"))
fr, en = stsb("fr"), stsb("en")
assert len(fr) == len(en)
print(len(fr), "paires"); fr.head(3)

In [ ]:
def encoder(modele, phrases, prefixe=""):
    return modele.encode([prefixe + p for p in phrases], batch_size=64, convert_to_numpy=True,
                         normalize_embeddings=True, show_progress_bar=False)

def sts(modele, s1, s2, gold, prefixe=""):
    a, b = encoder(modele, s1, prefixe), encoder(modele, s2, prefixe)
    return 100 * spearmanr((a * b).sum(1), gold)[0]

def vitesse(modele, phrases, prefixe=""):
    encoder(modele, phrases[:64], prefixe)
    if DEVICE == "cuda": torch.cuda.synchronize()
    t0 = time.perf_counter(); encoder(modele, phrases, prefixe)
    if DEVICE == "cuda": torch.cuda.synchronize()
    return len(phrases) / (time.perf_counter() - t0)

phrases_vitesse = (fr.sentence1.tolist() + fr.sentence2.tolist())[:2000]
lignes = []
for nom in ["MIND v2"] + BASELINES:
    m = mind if nom == "MIND v2" else SentenceTransformer(nom, device=DEVICE)
    p = PREFIXES.get(nom, "")
    lignes.append({
        "modèle": nom,
        "paramètres (M)": sum(x.numel() for x in m.parameters()) / 1e6,
        "dim": getattr(m, "get_embedding_dimension", None) and m.get_embedding_dimension() or m.get_sentence_embedding_dimension(),
        "STSb fr": sts(m, fr.sentence1, fr.sentence2, fr.similarity_score, p),
        "STSb en": sts(m, en.sentence1, en.sentence2, en.similarity_score, p),
        "STSb fr↔en": sts(m, fr.sentence1, en.sentence2, fr.similarity_score, p),
        "phrases/s": vitesse(m, phrases_vitesse, p),
    })
    if m is not mind:
        del m; gc.collect(); torch.cuda.empty_cache() if DEVICE == "cuda" else None
partie_a = pd.DataFrame(lignes).set_index("modèle")
partie_a.to_csv(f"{SORTIE}/bench4_partie_a.csv")
partie_a.round(1)

### A2. Exemples concrets de recherche sémantique
Petit test qualitatif : pour chaque requête, la phrase la plus proche dans un mini-corpus.

In [ ]:
corpus = [
    "Le train pour Lyon part à 8 h de la gare de Lyon.", "Il pleut depuis trois jours sur la Bretagne.",
    "La recette demande 200 g de farine et trois œufs.", "Le PSG a gagné le match 3-1 hier soir.",
    "Mon ordinateur portable ne démarre plus.", "La Banque centrale a relevé ses taux d'intérêt.",
    "Les chats dorment environ quinze heures par jour.", "Le musée du Louvre est fermé le mardi.",
]
requetes = ["À quelle heure est le TGV ?", "Quel temps fait-il à Rennes ?", "Comment faire un gâteau ?",
            "Résultat du foot", "Mon PC est en panne", "L'inflation et la politique monétaire", "Combien dort un chat ?"]
for nom, m in [("MIND v2", mind)]:
    c, q = encoder(m, corpus), encoder(m, requetes)
    for r, s in zip(requetes, q @ c.T):
        print(f"{r:45s} -> {corpus[int(s.argmax())]}  ({s.max():.2f})")

## Partie B — MTEB français
Les résultats sont mis en cache dans `mteb_cache/` : si la session coupe, relancer reprend où ça s'était arrêté.

In [ ]:
from mteb.cache import ResultCache
CACHE = ResultCache(f"{SORTIE}/mteb_cache")
NOMS = [t.metadata.name for t in mteb.get_benchmark("MTEB(fra, v1)").tasks]
if LEGER:
    NOMS = ["SICKFr", "STS22", "STSBenchmarkMultilingualSTS", "PawsXPairClassification", "OpusparcusPC",
            "SyntecRetrieval", "SyntecReranking", "AlloProfClusteringS2S", "MTOPIntentClassification", "AmazonReviewsClassification"]
TACHES = mteb.get_tasks(tasks=NOMS, languages=["fra"], exclusive_language_filter=True, exclude_superseded=False)
DANS_LE_DOMAINE = {"PawsXPairClassification", "XNLI"}   # vues (train) pendant l'entraînement de MIND
print(len(TACHES), "tâches :", [t.metadata.name for t in TACHES])

In [ ]:
def charger(nom):
    if nom == "MIND v2":
        return mind
    try:
        return mteb.get_model(nom)  # gère les préfixes (e5 : "query: " / "passage: ")
    except Exception as e:
        print("mteb.get_model a échoué, SentenceTransformer direct :", e)
        return SentenceTransformer(nom, device=DEVICE)

scores = {}
for nom in ["MIND v2"] + BASELINES:
    t0 = time.time()
    m = charger(nom)
    r = mteb.evaluate(m, TACHES, cache=CACHE, raise_error=False, encode_kwargs={"batch_size": 64})
    s = {}
    for tr in r.task_results:  # les tâches sont déjà restreintes aux sous-ensembles français
        s[tr.task_name] = 100 * tr.get_score()
    scores[nom] = s
    print(f"{nom} : {len(s)} tâches en {(time.time() - t0) / 60:.1f} min", "| erreurs :", list(r.exceptions or [])[:3])
    if m is not mind:
        del m; gc.collect(); torch.cuda.empty_cache() if DEVICE == "cuda" else None

In [ ]:
res = pd.DataFrame(scores)
types = {t.metadata.name: t.metadata.type for t in TACHES}
res.insert(0, "type", [types.get(i, "?") for i in res.index])
res.index = [f"{i} ⚠️ (vu à l'entraînement)" if i in DANS_LE_DOMAINE else i for i in res.index]
res = res.sort_values(["type"])
res.to_csv(f"{SORTIE}/bench4_mteb_fr_detail.csv")
modeles = [c for c in res.columns if c != "type"]
res.style.format("{:.1f}", subset=modeles).background_gradient(axis=1, cmap="RdYlGn", subset=modeles)

In [ ]:
# moyenne par type de tâche, puis moyenne des types (comme le leaderboard MTEB) — sans les tâches vues à l'entraînement
hors = res[~res.index.str.contains("⚠️")]
par_type = hors.groupby("type")[modeles].mean()
par_type.loc["MOYENNE (types)"] = par_type.mean()
par_type.loc["MOYENNE (tâches)"] = hors[modeles].mean()
par_type.to_csv(f"{SORTIE}/bench4_mteb_fr_par_type.csv")
display(par_type.T.sort_values("MOYENNE (types)", ascending=False).round(1))

ax = par_type.loc["MOYENNE (types)"].sort_values().plot.barh(figsize=(8, 4), color=["#c55" if m == "MIND v2" else "#58a" for m in par_type.loc["MOYENNE (types)"].sort_values().index])
ax.set_title("MTEB français — moyenne des types de tâches (hors tâches vues)"); ax.set_xlabel("score")
plt.tight_layout(); plt.savefig(f"{SORTIE}/bench4_mteb_fr.png", dpi=120); plt.show()

### Comment lire les résultats
- **MIND vs Geotrend (sans fine-tuning)** : ce que ton entraînement apporte vraiment.
- **MIND vs distiluse / MiniLM** : modèles de taille proche, entraînés sur beaucoup plus de données → la cible réaliste à battre.
- MIND a été entraîné sur des **paires de paraphrases/NLI** (phrases courtes) : attends-toi à de bons scores en STS / paires,
  et plus faibles en **recherche de documents** (Alloprof, BSARD, Syntec) où les textes sont longs et le couple question→passage différent.
- Dim 256 vs 384–768 : MIND est **2–3× plus compact en stockage** d'index, un vrai avantage à mettre en avant.